In [1]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

/Users/jobinwattasseril/Projects/TaSeSum/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MAX_WORDS_PER_CHUNK = 100


def filter_segments_by_duration(
    segments: list[dict], duration_threshold: float = 2
) -> list[dict]:
    filtered_segments = []
    for segment in segments:
        if (segment["end"] - segment["start"]) < duration_threshold:
            continue
        filtered_segments.append(segment)
    return filtered_segments


def group_segments_into_chunks(segments: list[dict]) -> list[list[dict]]:
    word_count = 0
    grouped_segments = []
    chunk = []
    for segment in segments:
        num_words = len(segment["text"].split(" "))
        if (word_count + num_words) <= MAX_WORDS_PER_CHUNK:
            word_count += num_words
            chunk.append(segment)
        else:
            grouped_segments.append(chunk)
            chunk = []
            word_count = 0
    return grouped_segments


def merge_segments_in_chunk(chunk: list[dict]) -> dict:
    merged_segment = {}
    merged_text = ""
    merged_words = []
    no_speech_probs = []
    for segment in chunk:
        merged_text += segment["text"]
        merged_words.append(segment["words"])
        no_speech_probs.append(segment["no_speech_prob"])
    merged_segment["text"] = merged_text
    merged_segment["start"] = chunk[0]["start"]
    merged_segment["end"] = chunk[-1]["end"]
    merged_segment["words"] = merged_words
    merged_segment["no_speech_prob"] = no_speech_probs
    return merged_segment


def preprocess_segments(segments: list[dict]) -> list[dict]:
    grouped_segments = group_segments_into_chunks(segments)
    merged_segments = []
    for chunk in grouped_segments:
        merged_segment = merge_segments_in_chunk(chunk)
        merged_segments.append(merged_segment)
    # merged_segments = merge_segments(segments)
    # filtered_segments = filter_segments_by_duration(merged_segments)
    return merged_segments

In [3]:
import pickle

with open("/Users/jobinwattasseril/Projects/TaSeSum/test_data/result_dict_cuban.pickle", "rb") as f:
    result_dict = pickle.load(f)

In [4]:
segments = result_dict["segments"]
segments = preprocess_segments(segments)

In [13]:
docs = [segment["text"].strip() for segment in segments]

In [14]:
docs

['the person who controls the algorithm controls the world. Right? And if you are committed to one specific platform as your singular source of information or affiliated platforms, then whoever controls the algorithm or the programming there controls you. The following is a conversation with Mark Cuban, a multibillionaire business man, an investor and star of the series Shark Tank, long time principle owner of the Dallas Mavericks, and is someone who is unafraid to get into frequent battles on X. Most recently, orthopics of DEI,',
 "and identity politics with the likes of Elon Musk and Jordan Peterson. This is the Lex Freedman podcast to support it. We should cut our sponsors in the description. And now, dear friends, here's Mark Cuban. You've started many businesses invested in many businesses, heard a lot of pitches privately and on Shark Tank. So you're the perfect person to ask what makes a great entrepreneur. Somebody who's curious, they want to keep on learning because business i

In [15]:
representation_model = KeyBERTInspired()
topic_model = BERTopic(representation_model=representation_model)

In [16]:
topics, probs = topic_model.fit_transform(docs)

In [17]:
topics, probs

([2,
  1,
  1,
  1,
  -1,
  -1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  -1,
  5,
  -1,
  -1,
  -1,
  1,
  1,
  0,
  1,
  -1,
  -1,
  1,
  -1,
  1,
  1,
  1,
  1,
  1,
  1,
  -1,
  1,
  1,
  2,
  1,
  1,
  1,
  1,
  5,
  0,
  1,
  1,
  1,
  -1,
  0,
  0,
  1,
  3,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  -1,
  -1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  1,
  -1,
  -1,
  -1,
  -1,
  1,
  1,
  1,
  -1,
  1,
  1,
  -1,
  0,
  0,
  -1,
  3,
  -1,
  0,
  1,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  5,
  2,
  2,
  2,
  3,
  4,
  -1,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  -1,
  4,
  4,
  2,
  2,
  4,
  4,
  2,
  2,
  2,
  4,
  4,
  2,
  0,
  0,
  1,
  0,
  0,
  -1,
  -1,
  0,
  0,
  0,
  -1,
  -1,
  0,
  0,
  0,
  0,
  2,
  2,
  2,
  2,
  2,
  -1,
  -1,
  2,
  -1,
  -1,
  -1,
  -1,
  2,
  -1,
  3,
  3,
  -1,
  -1,
  -1,
  2,
  2,
  -1,
  2,
  2,
  -1,
  -1,
  2,
  2,
  2,
  2,
  2,
  2,
  5,
  2,
  2,
  2,


In [18]:
len(topics)

250

In [19]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,51,-1_war_impressions_world_me,"[war, impressions, world, me, people, money, r...",[And you have to decide are they going to step...
1,0,52,0_technology_company_models_software,"[technology, company, models, software, server...",[right? And they do it on the front end with p...
2,1,48,1_entrepreneur_entrepreneurship_business_start,"[entrepreneur, entrepreneurship, business, sta...","[what did you say? I mean, it's a terrifying t..."
3,2,37,2_ideology_trump_biden_speech,"[ideology, trump, biden, speech, media, woke, ...",[I guess another way to say that is they don't...
4,3,35,3_pharmacies_pharmacy_medicare_healthcare,"[pharmacies, pharmacy, medicare, healthcare, c...","[I think it was like 7,500% or increased a low..."
5,4,14,4_ideology_organizations_universities_discrimi...,"[ideology, organizations, universities, discri...",[DEI programs that universities and companies ...
6,5,13,5_nba_basketball_stadium_arena,"[nba, basketball, stadium, arena, fans, sport,...","[It was like oh wow. Okay fun. Yeah, the energ..."


In [25]:
topic_model.get_document_info(docs)["Topic"].value_counts()

Topic
 0    52
-1    51
 1    48
 2    37
 3    35
 4    14
 5    13
Name: count, dtype: int64

# For Integration

In [1]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

/home/jobin/Projects/transcript_summarizer/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MAX_WORDS_PER_CHUNK = 100

def filter_segments_by_duration(
    segments: list[dict], duration_threshold: float = 2
) -> list[dict]:
    filtered_segments = []
    for segment in segments:
        if (segment["end"] - segment["start"]) < duration_threshold:
            continue
        filtered_segments.append(segment)
    return filtered_segments


def group_segments_into_chunks(segments: list[dict]) -> list[list[dict]]:
    word_count = 0
    grouped_segments = []
    chunk = []
    for segment in segments:
        num_words = len(segment["text"].split(" "))
        if (word_count + num_words) <= MAX_WORDS_PER_CHUNK:
            word_count += num_words
            chunk.append(segment)
        else:
            grouped_segments.append(chunk)
            chunk = []
            word_count = 0
    return grouped_segments

def merge_segments_in_chunk(chunk: list[dict]) -> dict:
    merged_segment = {}
    merged_text = ""
    merged_words = []
    no_speech_probs = []
    for segment in chunk:
        merged_text += segment["text"]
        merged_words.append(segment["words"])
        no_speech_probs.append(segment["no_speech_prob"])
    merged_segment["text"] = merged_text
    merged_segment["start"] = chunk[0]["start"]
    merged_segment["end"] = chunk[-1]["end"]
    merged_segment["words"] = merged_words
    merged_segment["no_speech_prob"] = no_speech_probs
    return merged_segment

def preprocess_segments(segments: list[dict]) -> list[dict]:
    grouped_segments = group_segments_into_chunks(segments)
    merged_segments = []
    for chunk in grouped_segments:
        merged_segment = merge_segments_in_chunk(chunk)
        merged_segments.append(merged_segment)
    return merged_segments

In [27]:
def load_topic_model() -> BERTopic:
    representation_model = KeyBERTInspired()
    topic_model = BERTopic(representation_model=representation_model)
    return topic_model

def add_topics_to_segments(segments: list[dict], topic_model: BERTopic) -> list[dict]:
    segments_with_topics = []
    docs = [segment["text"].strip() for segment in segments]
    topics, _ = topic_model.fit_transform(docs)

    for topic_id, segment in zip(topics, segments):
        segment_with_topic = segment
        segment_with_topic["topic_id"] = topic_id
        segments_with_topics.append(segment_with_topic)
    return segments_with_topics

def get_topics_dict_wo_prob(topic_model) -> dict[int]:
    topics_dict = topic_model.get_topics()
    topics_dict_wo_prob = {}
    for topic_id in topics_dict:
        topic_prob_list = topics_dict[topic_id]
        topics = [x[0] for x in topic_prob_list]
        topics_dict_wo_prob[topic_id] = topics
    return topics_dict_wo_prob

    

In [4]:
import pickle

with open("../data/pickled/result_dict_cuban.pickle","rb") as f:
    result_dict = pickle.load(f)

segments = result_dict["segments"]
segments = preprocess_segments(segments)

In [5]:
segments = result_dict["segments"]
segments = preprocess_segments(segments)

In [6]:
topic_model = load_topic_model()

In [7]:
segments_with_topics = add_topics_to_segments(segments, topic_model)

In [28]:
get_topics_dict_wo_prob(topic_model)

{-1: ['war',
  'impressions',
  'world',
  'me',
  'people',
  'money',
  'really',
  'they',
  'you',
  'feel'],
 0: ['technology',
  'company',
  'models',
  'software',
  'server',
  'stock',
  'google',
  'yahoo',
  'me',
  'they'],
 1: ['entrepreneur',
  'entrepreneurship',
  'business',
  'start',
  'success',
  'work',
  'money',
  'say',
  'do',
  'dont'],
 2: ['ideology',
  'trump',
  'biden',
  'speech',
  'media',
  'woke',
  'immigration',
  'has',
  'immigrants',
  'most'],
 3: ['pharmacies',
  'pharmacy',
  'medicare',
  'healthcare',
  'companies',
  'drug',
  'drugs',
  'insurance',
  'business',
  'medication'],
 4: ['ideology',
  'organizations',
  'universities',
  'discrimination',
  'corporate',
  'criticize',
  'criticizing',
  'racism',
  'leadership',
  'companies'],
 5: ['nba',
  'basketball',
  'stadium',
  'arena',
  'fans',
  'sport',
  'ball',
  'players',
  'game',
  'balls']}

In [1]:
from bertopic import BERTopic

# Sample data
documents = [
    "The economy is growing rapidly, with increased investment in technology.",
    "Advances in AI and machine learning are transforming industries.",
    "The healthcare sector is seeing innovations due to new medical technologies.",
    "Political debates focus on the economic policies and their impacts.",
    "New developments in renewable energy sources are crucial for sustainability.",
]

# Initialize BERTopic
topic_model = BERTopic()

# Fit the model on the documents
topics, probs = topic_model.fit_transform(documents)

# Print the topics
for idx, topic in enumerate(topics):
    print(f"Document {idx} is about topic {topic}")

# Get the topic representations
topic_info = topic_model.get_topic_info()
print(topic_info)

# Get the most representative words for each topic
for topic_id in set(topics):
    print(f"Topic {topic_id}: {topic_model.get_topic(topic_id)}")

/home/jobin/Projects/TaSeSum/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jobin/Projects/TaSeSum/venv/lib/python3.10/site-packages/umap/spectral.py:521: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  eigenvalues, eigenvectors = scipy.sparse.linalg.eigsh(
/home/jobin/Projects/TaSeSum/venv/lib/python3.10/site-packages/umap/spectral.py:521: RuntimeWarning: k >= N for N * N square matrix. Attempting to use scipy.linalg.eigh instead.
  eigenvalues, eigenvectors = scipy.sparse.linalg.eigsh(


TypeError: Cannot use scipy.linalg.eigh for sparse A with k >= N. Use scipy.linalg.eigh(A.toarray()) or reduce k.